In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pickagm.distributions import ensemble_ks_bounds

from phd_project.config.config import load_config 
from phd_project.scripts.cache_utils import fingerprint, load_or_compute
from phd_project.scripts.WP1_ground_motion_set.gm_selection import (
    calculate_gcim_distributions_for_sites,
    _selection_ctx_fingerprint_inputs,
    stripe_input_fingerprint,
    find_stale_stripes,
)

from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA06_gm_selection import (
    setup_AvgSA06_gcim_gm_selection,
    SELECTION_CONFIG,
    stripe_source_fps,
)

cfg = load_config()

# 034 — GCIM distributions, AvgSA([0, 6])

Computes the GCIM (Generalised Conditional Intensity Measure) target distributions for every
`(site, iml)` disaggregation, conditioned on **AvgSA([0, 6])**. For each site/intensity level the
GMMs and correlation models are run over that site's disaggregation to produce the target
`stats` / `pdfs` / `cdfs`. This is the slow target-building step that precedes record selection.

**Upstream** — read by `setup_AvgSA06_gcim_gm_selection()` (`setup_AvgSA06_gm_selection.py`):

| Input | Config key |
|---|---|
| IML-based disaggregations (nb 021) | `cfg["proc_data"]["AvgSA_06_disagg_data_gm_selection"]` |
| Per-(site, imt, iml) poe stats | `cfg["proc_data"]["AvgSA_06_disagg_stats_gm_selection"]` |
| Per-site IML subset (one MSA stripe each) | `cfg["proc_data"]["AvgSA_06_imls_for_selection"]` |
| Site model | `cfg["hazard_models"]["eshm20_wp1_site_model"]` |
| Median-branch GMM logic tree | `cfg["hazard_models"]["eshm20_AvgSA_06_median_lt"]` |
| Correlation flatfiles + GM database | `cfg["proc_data"]["corr_model"]`, `cfg["proc_data"]["gm_database"]` (db used for selection, not the gcim calc) |

**Downstream** — `gcim_dist_AvgSA_06.pickle` is consumed by
`035-gm_selection_AvgSA_06_stage1_compute.ipynb`, which fingerprints it as `gcim_file`, so the
manifest written here closes the provenance chain into Stage-1.

**Output** — `cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_06.pickle"`, plus a
`gcim_dist_AvgSA_06.pickle.manifest.json` provenance sidecar recording a content hash of every
input (upstream files, selection context, percentiles, `pickagm` version) along with the git
commit and timestamp.

**Run order** — run top to bottom. The compute cell is provenance-cached via
`cache_utils.load_or_compute`: a re-run with unchanged inputs reloads the pickle instantly
(`[cache] ... loaded (inputs match).`) instead of recomputing. Set `FORCE_RECOMPUTE = True` to
rebuild and overwrite regardless of the cache.

In [ ]:
# set up the gcim / record selection
site_iml_disaggs, disagg_stats, site_model, basic_selection_ctx, _ = setup_AvgSA06_gcim_gm_selection()
percentiles = SELECTION_CONFIG["percentiles"]

# Incremental cache: only (re)build gcim for stripes that are missing or stale on
# disk. A (site, iml) stripe is valid iff its result pickle + .manifest.json match
# its per-stripe fingerprint (which EXCLUDES the IML subset JSON), so appending IMLs
# to the "union" lists makes only the NEW (site, iml) stale.
RESULT_FOLDER = cfg["results"]["AvgSA_06_record_selection"]
source_fps_stripe = stripe_source_fps()
wanted = list(site_iml_disaggs.keys())
fp_fn = lambda s, i: stripe_input_fingerprint(
    s, i, source_fps_stripe, basic_selection_ctx, SELECTION_CONFIG)
to_compute, valid = find_stale_stripes(wanted, RESULT_FOLDER, fp_fn)
print(f"{len(wanted)} wanted (site, iml): {len(valid)} already valid, "
      f"{len(to_compute)} to (re)compute.")

In [ ]:
# Escape hatch: force-recompute gcim for ALL wanted stripes, ignoring the per-stripe
# manifests. Leave False for normal incremental runs (only stale/new stripes rebuild).
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE:
    to_compute = list(wanted)

gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_06.pickle"

In [ ]:
# Compute the GCIM distributions ONLY for the stale/new (site, iml), then MERGE them
# into the existing gcim pickle so it stays the complete set — nothing already computed
# is dropped, only the stale/new keys are (re)written. 035 consumes it immediately.
batch = {k: site_iml_disaggs[k] for k in to_compute}
if batch:
    new_gcim = calculate_gcim_distributions_for_sites(
        batch, disagg_stats, site_model, basic_selection_ctx, percentiles)

    gcim_dists = {}
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as f:
            gcim_dists = pickle.load(f)
    gcim_dists.update(new_gcim)   # overwrite only the recomputed keys

    gcim_dist_fp.parent.mkdir(parents=True, exist_ok=True)
    with open(gcim_dist_fp, "wb") as f:
        pickle.dump(gcim_dists, f)
    print(f"(Re)computed gcim for {len(new_gcim)} (site, iml); "
          f"pickle now holds {len(gcim_dists)} total -> {gcim_dist_fp.name}")
else:
    # Nothing stale: load the existing complete gcim (if any) so later cells still work.
    if gcim_dist_fp.is_file():
        with open(gcim_dist_fp, "rb") as f:
            gcim_dists = pickle.load(f)
    else:
        gcim_dists = {}
    print("All stripes valid — no gcim to compute. Proceed to 035/036 to refresh CSVs if needed.")